In [11]:
import pandas as pd
import re

# 1. Load and clean messy data
df = pd.read_csv('/home/emchatwin/Kelly Lab/ms_qc_processor/input2/bachelorette.csv', skiprows=[1])

elim_cols = [f'ELIM_{i}' for i in range(1, 11)]
date_cols = [f'DATE_{i}' for i in range(1, 11)]
df.columns = ['SHOW', 'SEASON', 'CONTESTANT'] + elim_cols + date_cols
df = df[df['SEASON'] != 'SEASON'].copy()

# Elimination codes
elim_codes = {'E', 'ED', 'EU', 'EQ', 'EF'}

def get_elim_week(row):
    for w in range(1, 11):
        val = row[f'ELIM_{w}']
        if val in elim_codes:
            return w
        if val == 'W':
            return 11
    return 11

df['ELIM_WEEK'] = df.apply(get_elim_week, axis=1)

# First impression rose (week 1 only)
df['FIRST_IMPRESSION_ROSE'] = df['ELIM_1'] == 'R1'


# 2. Helper to parse date code and get group size
def parse_date_code(code):
    if pd.isna(code) or str(code).strip() == '':
        return None, False
    code_str = str(code).strip()
    first_code = code_str.split(',')[0].strip()
    match = re.match(r'D(\d+)', first_code)
    if match:
        size = int(match.group(1))
        return size, (size == 1)
    return 1, True  # fallback

# 3. Build weekly timeline with roses
records = []
for _, row in df.iterrows():
    show = row['SHOW']
    season = row['SEASON']
    contestant = row['CONTESTANT']
    elim_week = row['ELIM_WEEK']
    fir = row['FIRST_IMPRESSION_ROSE']
    for week in range(1, 11):
        alive = week < elim_week                    # alive at start of week
        weekly_rose = elim_week > week              # got rose at end of week
        date_val = row[f'DATE_{week}']
        group_size, is_one_on_one = parse_date_code(date_val)
        has_date = group_size is not None
        date_rose = is_one_on_one                   # one-on-one always gives a date rose
        date_ratio = 1.0 / group_size if has_date else 0.0
        records.append({
            'SHOW': show,
            'SEASON': season,
            'CONTESTANT': contestant,
            'WEEK': week,
            'ALIVE': alive,
            'HAS_DATE': has_date,
            'ONE_ON_ONE': is_one_on_one,
            'DATE_GROUP_SIZE': group_size if has_date else 0,
            'DATE_RATIO': date_ratio,
            'DATE_ROSE': date_rose,
            'WEEKLY_ROSE': weekly_rose,
            'FIRST_IMPRESSION_ROSE': (week == 1 and fir),
            'ELIMINATED_THIS_WEEK': (week == elim_week and elim_week <= 10)
        })

weekly_df = pd.DataFrame(records)


weekly_df.to_csv('contestant_weekly.csv', index=False)

print("Files saved:")
print("  - contestant_weekly.csv")
print("  - weekly_dates_summary.csv")

Files saved:
  - contestant_weekly.csv
  - weekly_dates_summary.csv
